In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
import pandas as pd
from sqlalchemy import create_engine

env_path = Path('../../.env')
load_dotenv(dotenv_path=env_path)

USER = os.getenv("MYSQL_USER")
PASSWORD = os.getenv("MYSQL_PASSWORD")
DATABASE = os.getenv("MYSQL_DATABASE")
HOST = "192.168.86.4"
PORT = "3306"

engine = create_engine(f"mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

df = pd.read_sql("SELECT * FROM auctions", engine)

os.makedirs('./db_export', exist_ok=True)

# 2. Pre-process for Parquet stability
# We explicitly convert 'object' columns (like UUIDs/NBT strings) to standard strings
# to prevent encoding hiccups.
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)

print("Converting to Native PyArrow Table...")
# 3. Convert DataFrame to PyArrow Table
# This bypasses the pandas.to_parquet() wrapper entirely
table = pa.Table.from_pandas(df, preserve_index=False)

print("Writing to Parquet with Snappy compression...")
# 4. Save the file
# Snappy is much faster for your Ryzen 9 to process than Gzip
pq.write_table(table, './db_export/sb_auctions_2026.parquet', compression='snappy')

print(f"Export Success! File saved to ./db_export/sb_auctions_2026.parquet")

Converting to Native PyArrow Table...
Writing to Parquet with Snappy compression...
Export Success! File saved to ./db_export/sb_auctions_2026.parquet


In [ ]:
preview_file = './db_export/preview_sample.csv'

df.head(1000).to_csv(preview_file, index=False)

print(f"Preview generated: {preview_file}")
print(f"File size: {os.path.getsize(preview_file) / 1024:.2f} KB")